# OpenVSP 4-view screenshot

`.vsp3`モデルを4方向から撮影し、1枚のPNGへ合成します。OpenVSPのGUIは専用の子プロセスで一時的に起動し、撮影終了後に閉じます。そのため、このNotebookのカーネル内で読み込まれているOpenVSPモデルには影響しません。

必要なPython環境には、GUI対応のOpenVSP Python APIとPillowが必要です。描画方式は `preserve`、`wire`、`hidden`、`shade`、`texture` から選べます。

In [ ]:
from pathlib import Path
import sys

from IPython.display import Image as NotebookImage, display

start_dir = Path.cwd().resolve()
repo_root = next(
    (
        candidate
        for candidate in [start_dir, *start_dir.parents]
        if (candidate / "src" / "OpenVSPViewCapture.py").exists()
    ),
    None,
)
if repo_root is None:
    raise FileNotFoundError("Could not find the repository root.")
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from src.OpenVSPViewCapture import (  # noqa: E402
    ORTHOGRAPHIC_FOUR_VIEW_LAYOUT,
    ViewSpec,
    capture_vsp3_views,
)

print("repo_root:", repo_root)

## Capture

配置は、左上が上面図、右上が左側面図、左下が正面図、右下が左鳥瞰図です。上面図と左側面図は、機首が画面下を向くように撮影後の画像を反時計回りに90度回転します。

In [ ]:
model_name = "Cessna-210"
# model_name = "Boeing_777-9x_mod"
# model_name = "G103A"
model_path = repo_root / "examples" / "models" / f"{model_name}" / f"{model_name}.vsp3"
output_path = repo_root / "examples" / "notebooks" / "view_capture" / "output" / f"{model_name}_four_views.png"

result_path = capture_vsp3_views(
    model_path,
    output_path,
    size=(1600, 1200),
    view_grid=ORTHOGRAPHIC_FOUR_VIEW_LAYOUT,
    render_mode="hidden",
    fit=True,
    fit_to_content=True,
    content_padding_px=12,
    transparent_background=False,
    show_axis=False,
    show_borders=False,
    overwrite=True,
)

display(NotebookImage(filename=str(result_path)))
result_path

## Custom layout

`ViewSpec(view, rotate_deg)`を並べると、行数・列数・方向・撮影後の90度単位の回転を変更できます。`rotate_deg`の正値は反時計回りです。

In [ ]:
custom_grid = (
    (ViewSpec("top", 90), ViewSpec("left", 90)),
    (ViewSpec("front"), ViewSpec("left_iso")),
)
custom_grid